# Gradio UI — Interactive RAG Demo

A thin Gradio wrapper around the retrieval and generation pipeline.
The UI deliberately reflects backend quality — no answer smoothing,
no fallback messages. If retrieval is weak, the demo will show it.

The chat function calls `retrieve()` and `answer_from_chunks()` directly
so sources can be appended to every response.

## 1. Setup

In [ ]:
import sys
import os
import pickle
from pathlib import Path
from openai import OpenAI

sys.path.insert(0, str(Path("..").resolve()))
from src.retriever import Retriever
from src.generator import answer_from_chunks

CHROMA_DIR = "../chroma_db"
COLLECTION  = "finance_rag"
CACHE_PATH  = Path("../chunks_cache.pkl")
LLM_MODEL   = "llama-3.3-70b-versatile"

with open(CACHE_PATH, "rb") as f:
    all_chunks = pickle.load(f)

retriever = Retriever.from_index(CHROMA_DIR, COLLECTION, all_chunks)
client    = OpenAI(
    api_key=os.environ["GROQ_API_KEY"],
    base_url="https://api.groq.com/openai/v1",
)
print("Retriever and LLM client ready.")

## 2. Chat Function

`chat_fn` follows the Gradio `ChatInterface` contract: it receives the
current message and the full conversation history, and returns a string.

We retrieve once and pass the chunks directly to `answer_from_chunks` so
the same chunk list is used for both the answer and the source attribution.
Calling `answer()` instead would retrieve a second time — wasteful and
potentially inconsistent.

In [ ]:
import gradio as gr


def chat_fn(message: str, history: list) -> str:
    # history is a list of [user, bot] pairs — available if you want
    # multi-turn context, but single-turn retrieval is fine for a RAG demo.
    chunks   = retriever.retrieve(message, top_k=5)
    response = answer_from_chunks(message, chunks, client, LLM_MODEL)

    sources    = sorted({c.source for c in chunks})
    sources_md = "\n".join(f"- {s}" for s in sources)
    return f"{response}\n\n**Sources:**\n{sources_md}"


print("chat_fn defined.")

## 3. Launch

Example questions are drawn from the eval set so the demo starts with
queries that are known to retrieve correctly.

In [ ]:
demo = gr.ChatInterface(
    fn=chat_fn,
    title="Finance RAG",
    description="Ask questions about SEC filings and earnings reports from the corpus.",
    examples=[
        "What were Amazon's total net sales for fiscal year 2025?",
        "What was NVIDIA's Data Center revenue for fiscal year 2026?",
        "By what percentage did Meta's revenue grow in fiscal year 2025?",
        "What was Tesla's net income for fiscal year 2025?",
        "What is NVIDIA's current stock price?",
    ],
)

demo.launch(share=False)  # share=True for a public Gradio URL (72 h)